In [1]:
import os
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
clinical_file = "clinical_feature_engineered_data.xlsx"

ranking_file = (
    "personalized_medicine_ranking_results.xlsx"
)

output_file = (
    "patient_risk_safety_layer_results.xlsx"
)

configuration_file = (
    "patient_risk_safety_configuration.json"
)

required_files = [
    clinical_file,
    ranking_file
]

for file_name in required_files:
    if not os.path.exists(file_name):
        raise FileNotFoundError(
            f"{file_name} was not found. "
            "Place it in the notebook folder."
        )

print("All required files are available.")

All required files are available.


In [3]:
clinical_data = pd.read_excel(
    clinical_file,
    sheet_name="Feature_Engineered_Data"
)

print("Clinical dataset:", clinical_data.shape)
print("\nClinical columns:")
print(clinical_data.columns.tolist())

display(clinical_data.head())

Clinical dataset: (500, 31)

Clinical columns:
['patient_id', 'age_years', 'sex', 'condition', 'glucose_mg_dl', 'systolic_bp_mmhg', 'diastolic_bp_mmhg', 'cholesterol_mg_dl', 'heart_rate_bpm', 'recorded_allergy', 'family_history', 'adherence_level', 'historical_medication_class', 'recorded_outcome', 'data_quality_status', 'age_group', 'pulse_pressure_mmhg', 'systolic_diastolic_ratio', 'glucose_cholesterol_interaction', 'glucose_age_interaction', 'cholesterol_age_interaction', 'bp_age_interaction', 'allergy_recorded_flag', 'family_history_flag', 'adherence_score', 'glucose_mg_dl_dataset_zscore', 'systolic_bp_mmhg_dataset_zscore', 'diastolic_bp_mmhg_dataset_zscore', 'cholesterol_mg_dl_dataset_zscore', 'heart_rate_bpm_dataset_zscore', 'measurement_deviation_score']


,patient_id,age_years,sex,condition,glucose_mg_dl,systolic_bp_mmhg,diastolic_bp_mmhg,cholesterol_mg_dl,heart_rate_bpm,recorded_allergy,...,bp_age_interaction,allergy_recorded_flag,family_history_flag,adherence_score,glucose_mg_dl_dataset_zscore,systolic_bp_mmhg_dataset_zscore,diastolic_bp_mmhg_dataset_zscore,cholesterol_mg_dl_dataset_zscore,heart_rate_bpm_dataset_zscore,measurement_deviation_score
0,SYN-0001,28,Female,Seasonal Allergy,97,132,85,214,73,None recorded,...,3696,0,0,2,-0.324226,0.254356,0.339338,0.424061,-0.315852,0.331566
1,SYN-0002,80,Female,Acid Reflux,80,123,79,189,59,None recorded,...,9840,0,1,1,-0.809337,-0.237734,-0.210049,-0.216205,-1.546215,0.603908
2,SYN-0003,36,Female,Asthma,86,121,66,192,82,None recorded,...,4356,0,0,1,-0.638121,-0.347087,-1.400387,-0.139373,0.475096,0.600013
3,SYN-0004,21,Male,High Cholesterol,103,120,69,278,85,None recorded,...,2520,0,1,2,-0.153010,-0.401764,-1.125693,2.063142,0.738745,0.896471
4,SYN-0005,58,Male,High Cholesterol,79,111,76,244,64,None recorded,...,6438,0,1,2,-0.837873,-0.893854,-0.484742,1.192380,-1.106799,0.903130


In [4]:
profile_recommendations = pd.read_excel(
    ranking_file,
    sheet_name="Profile_Recommendations"
)

ranking_long = pd.read_excel(
    ranking_file,
    sheet_name="Ranking_Long_Format"
)

print(
    "Profile recommendations:",
    profile_recommendations.shape
)

print(
    "Long-format rankings:",
    ranking_long.shape
)

display(ranking_long.head())

Profile recommendations: (100, 26)
Long-format rankings: (300, 7)


,test_profile_id,rank,recommended_medication_class,raw_recommendation_score,normalized_top_3_score,is_recorded_class,ranking_confidence_level
0,TEST-0001,1,Controller inhaler class,0.187092,0.352233,True,Low
1,TEST-0001,2,Acid-suppression class A,0.178393,0.335855,False,Low
2,TEST-0001,3,Acid-suppression class B,0.165676,0.311913,False,Low
3,TEST-0002,1,Antidiabetic class A,0.454071,0.528081,True,Moderate
4,TEST-0002,2,Antidiabetic class B,0.336010,0.390778,False,Moderate


In [5]:
required_clinical_columns = [
    "patient_id",
    "age_years",
    "condition",
    "glucose_mg_dl",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "cholesterol_mg_dl",
    "heart_rate_bpm",
    "recorded_allergy",
    "family_history",
    "adherence_level",
    "historical_medication_class"
]

missing_clinical_columns = [
    column
    for column in required_clinical_columns
    if column not in clinical_data.columns
]

if missing_clinical_columns:
    raise KeyError(
        "Missing clinical columns: "
        f"{missing_clinical_columns}"
    )

print("Required clinical columns verified.")

Required clinical columns verified.


In [6]:
RANDOM_STATE = 42

all_indices = np.arange(len(clinical_data))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=clinical_data[
        "historical_medication_class"
    ]
)

test_patient_profiles = (
    clinical_data
    .iloc[test_indices]
    .reset_index(drop=True)
)

print("Recovered test profiles:", test_patient_profiles.shape)
print("Ranking profiles:", len(profile_recommendations))

if len(test_patient_profiles) != len(profile_recommendations):
    raise ValueError(
        "The reconstructed test size does not match "
        "the recommendation dataset."
    )

Recovered test profiles: (100, 31)
Ranking profiles: 100


In [7]:
expected_classes = (
    test_patient_profiles[
        "historical_medication_class"
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)

ranking_actual_classes = (
    profile_recommendations[
        "actual_medication_class"
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)

class_matches = (
    expected_classes == ranking_actual_classes
)

print(
    "Matching target records:",
    class_matches.sum(),
    "out of",
    len(class_matches)
)

if not class_matches.all():
    mismatch_count = (~class_matches).sum()

    raise ValueError(
        f"Patient reconstruction failed for "
        f"{mismatch_count} records. Do not continue because "
        "recommendations may be connected to incorrect patients. "
        "Patient IDs must be exported directly from the original "
        "train/test split."
    )

print("Patient-to-recommendation alignment verified.")

Matching target records: 100 out of 100
Patient-to-recommendation alignment verified.


In [8]:
patient_identifiers = test_patient_profiles[
    "patient_id"
].to_numpy()

profile_recommendations["patient_id"] = (
    patient_identifiers
)

profile_to_patient = dict(
    zip(
        profile_recommendations["test_profile_id"],
        profile_recommendations["patient_id"]
    )
)

ranking_long["patient_id"] = (
    ranking_long["test_profile_id"]
    .map(profile_to_patient)
)

if ranking_long["patient_id"].isnull().any():
    raise ValueError(
        "Some recommendation rows could not be connected "
        "to a patient."
    )

print("Patient IDs attached successfully.")

Patient IDs attached successfully.


In [9]:
patient_safety = test_patient_profiles[
    required_clinical_columns
].copy()

patient_safety["recorded_allergy"] = (
    patient_safety["recorded_allergy"]
    .fillna("None recorded")
    .astype(str)
    .str.strip()
)

patient_safety["family_history"] = (
    patient_safety["family_history"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

patient_safety["adherence_level"] = (
    patient_safety["adherence_level"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

In [10]:
patient_safety["critical_glucose_flag"] = (
    (patient_safety["glucose_mg_dl"] < 54)
    |
    (patient_safety["glucose_mg_dl"] >= 300)
)

patient_safety["high_glucose_flag"] = (
    (patient_safety["glucose_mg_dl"] >= 200)
    &
    (~patient_safety["critical_glucose_flag"])
)

patient_safety["critical_bp_flag"] = (
    (patient_safety["systolic_bp_mmhg"] >= 180)
    |
    (patient_safety["diastolic_bp_mmhg"] >= 120)
)

patient_safety["low_bp_flag"] = (
    patient_safety["systolic_bp_mmhg"] < 90
)

patient_safety["high_cholesterol_flag"] = (
    patient_safety["cholesterol_mg_dl"] >= 240
)

patient_safety["heart_rate_review_flag"] = (
    (patient_safety["heart_rate_bpm"] < 50)
    |
    (patient_safety["heart_rate_bpm"] > 120)
)

patient_safety["older_adult_flag"] = (
    patient_safety["age_years"] >= 65
)

In [11]:
no_allergy_values = {
    "",
    "none",
    "none recorded",
    "no",
    "no known allergy",
    "no known allergies",
    "nan",
    "unknown"
}

patient_safety["allergy_review_flag"] = (
    ~patient_safety[
        "recorded_allergy"
    ]
    .str.lower()
    .isin(no_allergy_values)
)

patient_safety["low_adherence_flag"] = (
    patient_safety[
        "adherence_level"
    ]
    .str.lower()
    .isin([
        "low",
        "poor",
        "non-adherent",
        "nonadherent"
    ])
)

patient_safety["family_history_flag"] = (
    patient_safety[
        "family_history"
    ]
    .str.lower()
    .isin([
        "yes",
        "present",
        "positive",
        "true",
        "1"
    ])
)

In [12]:
patient_safety["patient_risk_score"] = (
    patient_safety["critical_glucose_flag"].astype(int) * 4
    +
    patient_safety["critical_bp_flag"].astype(int) * 4
    +
    patient_safety["low_bp_flag"].astype(int) * 3
    +
    patient_safety["heart_rate_review_flag"].astype(int) * 3
    +
    patient_safety["high_glucose_flag"].astype(int) * 2
    +
    patient_safety["high_cholesterol_flag"].astype(int) * 2
    +
    patient_safety["allergy_review_flag"].astype(int) * 2
    +
    patient_safety["low_adherence_flag"].astype(int) * 1
    +
    patient_safety["family_history_flag"].astype(int) * 1
    +
    patient_safety["older_adult_flag"].astype(int) * 1
)

In [13]:
def assign_patient_risk_level(row):
    if (
        row["critical_glucose_flag"]
        or row["critical_bp_flag"]
        or row["low_bp_flag"]
        or row["heart_rate_review_flag"]
    ):
        return "Critical Review"

    if row["patient_risk_score"] >= 5:
        return "High Review"

    if row["patient_risk_score"] >= 2:
        return "Moderate Review"

    return "Routine Review"


patient_safety["patient_risk_level"] = (
    patient_safety.apply(
        assign_patient_risk_level,
        axis=1
    )
)

print(
    patient_safety[
        "patient_risk_level"
    ].value_counts()
)

patient_risk_level
Moderate Review    52
Routine Review     42
Critical Review     3
High Review         3
Name: count, dtype: int64


In [14]:
def create_safety_reasons(row):
    reasons = []

    if row["critical_glucose_flag"]:
        reasons.append("Critical glucose review")

    if row["high_glucose_flag"]:
        reasons.append("High glucose review")

    if row["critical_bp_flag"]:
        reasons.append("Critical blood-pressure review")

    if row["low_bp_flag"]:
        reasons.append("Low blood-pressure review")

    if row["high_cholesterol_flag"]:
        reasons.append("High cholesterol review")

    if row["heart_rate_review_flag"]:
        reasons.append("Heart-rate review")

    if row["allergy_review_flag"]:
        reasons.append("Recorded allergy requires verification")

    if row["low_adherence_flag"]:
        reasons.append("Low adherence")

    if row["family_history_flag"]:
        reasons.append("Positive family history")

    if row["older_adult_flag"]:
        reasons.append("Older-adult medication review")

    if not reasons:
        return "No rule-based risk flags"

    return "; ".join(reasons)


patient_safety["safety_review_reasons"] = (
    patient_safety.apply(
        create_safety_reasons,
        axis=1
    )
)

display(
    patient_safety[
        [
            "patient_id",
            "condition",
            "patient_risk_score",
            "patient_risk_level",
            "safety_review_reasons"
        ]
    ].head(10)
)

,patient_id,condition,patient_risk_score,patient_risk_level,safety_review_reasons
0,SYN-0049,Asthma,0,Routine Review,No rule-based risk flags
1,SYN-0135,Type 2 Diabetes,0,Routine Review,No rule-based risk flags
2,SYN-0094,High Cholesterol,2,Moderate Review,High cholesterol review
3,SYN-0131,High Cholesterol,2,Moderate Review,High cholesterol review
4,SYN-0467,Seasonal Allergy,1,Routine Review,Low adherence
5,SYN-0430,Asthma,3,Moderate Review,Low adherence; Positive family history; Older-...
6,SYN-0241,Hypertension,8,Critical Review,Critical blood-pressure review; Recorded aller...
7,SYN-0447,Type 2 Diabetes,1,Routine Review,Older-adult medication review
8,SYN-0252,Seasonal Allergy,4,Moderate Review,Recorded allergy requires verification; Low ad...
9,SYN-0011,Hypertension,4,Moderate Review,Recorded allergy requires verification; Low ad...


In [15]:
disease_medicine_map = (
    clinical_data[
        [
            "condition",
            "historical_medication_class"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "condition": "disease",
            "historical_medication_class":
                "recommended_medication_class"
        }
    )
    .reset_index(drop=True)
)

disease_medicine_map[
    "disease_compatible"
] = True

display(disease_medicine_map)

,disease,recommended_medication_class,disease_compatible
0,Seasonal Allergy,Antihistamine class A,True
1,Acid Reflux,Acid-suppression class A,True
2,Asthma,Controller inhaler class,True
3,High Cholesterol,Lipid-lowering class B,True
4,Hypertension,Antihypertensive class B,True
5,Type 2 Diabetes,Antidiabetic class B,True
6,Asthma,Bronchodilator class,True
7,Hypertension,Antihypertensive class A,True
8,Seasonal Allergy,Antihistamine class B,True
9,High Cholesterol,Lipid-lowering class A,True


In [16]:
ranking_safety = ranking_long.merge(
    patient_safety,
    on="patient_id",
    how="left",
    validate="many_to_one"
)

ranking_safety = ranking_safety.merge(
    disease_medicine_map,
    left_on=[
        "condition",
        "recommended_medication_class"
    ],
    right_on=[
        "disease",
        "recommended_medication_class"
    ],
    how="left"
)

ranking_safety["disease_compatible"] = (
    ranking_safety[
        "disease_compatible"
    ]
    .fillna(False)
    .astype(bool)
)

print("Safety ranking shape:", ranking_safety.shape)

Safety ranking shape: (300, 34)


In [17]:
ranking_safety["eligibility_factor"] = np.where(
    ranking_safety["disease_compatible"],
    1.0,
    0.0
)

ranking_safety["safety_adjusted_score"] = (
    ranking_safety["raw_recommendation_score"]
    * ranking_safety["eligibility_factor"]
)

ranking_safety["requires_professional_review"] = (
    ranking_safety["allergy_review_flag"]
    |
    ranking_safety["critical_glucose_flag"]
    |
    ranking_safety["critical_bp_flag"]
    |
    ranking_safety["low_bp_flag"]
    |
    ranking_safety["heart_rate_review_flag"]
)

In [18]:
def assign_recommendation_status(row):
    if not row["disease_compatible"]:
        return "Blocked: disease incompatible"

    if row["requires_professional_review"]:
        return "Eligible: professional review required"

    if row["patient_risk_level"] == "High Review":
        return "Eligible: enhanced review required"

    if row["patient_risk_level"] == "Moderate Review":
        return "Eligible: standard safety review"

    return "Eligible: routine review"


ranking_safety["recommendation_safety_status"] = (
    ranking_safety.apply(
        assign_recommendation_status,
        axis=1
    )
)

In [19]:
ranking_safety = (
    ranking_safety
    .sort_values(
        [
            "patient_id",
            "safety_adjusted_score"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

ranking_safety[
    "safety_adjusted_rank"
] = (
    ranking_safety
    .groupby("patient_id")
    .cumcount()
    + 1
)

ranking_safety.loc[
    ~ranking_safety["disease_compatible"],
    "safety_adjusted_rank"
] = 0

ranking_safety[
    "safety_adjusted_rank"
] = ranking_safety[
    "safety_adjusted_rank"
].astype(int)

display(
    ranking_safety[
        [
            "patient_id",
            "condition",
            "recommended_medication_class",
            "raw_recommendation_score",
            "disease_compatible",
            "safety_adjusted_score",
            "safety_adjusted_rank",
            "recommendation_safety_status"
        ]
    ].head(15)
)

,patient_id,condition,recommended_medication_class,raw_recommendation_score,disease_compatible,safety_adjusted_score,safety_adjusted_rank,recommendation_safety_status
0,SYN-0007,Type 2 Diabetes,Antidiabetic class A,0.440807,True,0.440807,1,Eligible: standard safety review
1,SYN-0007,Type 2 Diabetes,Antidiabetic class B,0.312027,True,0.312027,2,Eligible: standard safety review
2,SYN-0007,Type 2 Diabetes,Antihistamine class A,0.033623,False,0.000000,0,Blocked: disease incompatible
3,SYN-0008,Asthma,Bronchodilator class,0.220804,True,0.220804,1,Eligible: professional review required
4,SYN-0008,Asthma,Acid-suppression class A,0.146892,False,0.000000,0,Blocked: disease incompatible
5,SYN-0008,Asthma,Antihistamine class A,0.133677,False,0.000000,0,Blocked: disease incompatible
6,SYN-0011,Hypertension,Antihypertensive class B,0.420714,True,0.420714,1,Eligible: professional review required
7,SYN-0011,Hypertension,Antihypertensive class A,0.161959,True,0.161959,2,Eligible: professional review required
8,SYN-0011,Hypertension,Antihistamine class B,0.067019,False,0.000000,0,Blocked: disease incompatible
9,SYN-0013,Seasonal Allergy,Antihistamine class B,0.197199,True,0.197199,1,Eligible: routine review


In [20]:
eligible_recommendations = ranking_safety[
    ranking_safety["disease_compatible"]
].copy()

final_recommendations = (
    eligible_recommendations
    .sort_values(
        [
            "patient_id",
            "safety_adjusted_score"
        ],
        ascending=[True, False]
    )
    .groupby("patient_id", as_index=False)
    .first()
)

final_recommendations = final_recommendations[
    [
        "patient_id",
        "condition",
        "recommended_medication_class",
        "safety_adjusted_score",
        "patient_risk_score",
        "patient_risk_level",
        "safety_review_reasons",
        "recommendation_safety_status"
    ]
]

final_recommendations = (
    final_recommendations.rename(
        columns={
            "recommended_medication_class":
                "final_rank_1_medication_class",
            "safety_adjusted_score":
                "final_rank_1_score"
        }
    )
)

display(final_recommendations.head(10))

,patient_id,condition,final_rank_1_medication_class,final_rank_1_score,patient_risk_score,patient_risk_level,safety_review_reasons,recommendation_safety_status
0,SYN-0007,Type 2 Diabetes,Antidiabetic class A,0.440807,3,Moderate Review,High glucose review; Positive family history,Eligible: standard safety review
1,SYN-0008,Asthma,Bronchodilator class,0.220804,3,Moderate Review,Recorded allergy requires verification; Older-...,Eligible: professional review required
2,SYN-0011,Hypertension,Antihypertensive class B,0.420714,4,Moderate Review,Recorded allergy requires verification; Low ad...,Eligible: professional review required
3,SYN-0013,Seasonal Allergy,Antihistamine class B,0.197199,0,Routine Review,No rule-based risk flags,Eligible: routine review
4,SYN-0028,High Cholesterol,Lipid-lowering class A,0.435825,5,High Review,High cholesterol review; Recorded allergy requ...,Eligible: professional review required
5,SYN-0039,Seasonal Allergy,Antihistamine class A,0.139959,1,Routine Review,Low adherence,Eligible: routine review
6,SYN-0049,Asthma,Controller inhaler class,0.187092,0,Routine Review,No rule-based risk flags,Eligible: routine review
7,SYN-0062,Type 2 Diabetes,Antidiabetic class A,0.512177,2,Moderate Review,High glucose review,Eligible: standard safety review
8,SYN-0065,Seasonal Allergy,Antihistamine class B,0.162648,1,Routine Review,Positive family history,Eligible: routine review
9,SYN-0073,Seasonal Allergy,Antihistamine class A,0.145040,1,Routine Review,Positive family history,Eligible: routine review


In [21]:
patients_with_recommendation = set(
    final_recommendations["patient_id"]
)

all_test_patients = set(
    patient_safety["patient_id"]
)

patients_without_recommendation = (
    all_test_patients
    - patients_with_recommendation
)

no_eligible_recommendations = (
    patient_safety[
        patient_safety["patient_id"].isin(
            patients_without_recommendation
        )
    ]
    .copy()
)

print(
    "Patients without an eligible recommendation:",
    len(no_eligible_recommendations)
)

Patients without an eligible recommendation: 4


In [22]:
safety_summary = pd.DataFrame({
    "metric": [
        "Total test patients",
        "Routine-review patients",
        "Moderate-review patients",
        "High-review patients",
        "Critical-review patients",
        "Patients with recorded-allergy review",
        "Disease-incompatible recommendations",
        "Recommendations requiring professional review",
        "Patients without eligible recommendation"
    ],
    "value": [
        len(patient_safety),

        (
            patient_safety["patient_risk_level"]
            == "Routine Review"
        ).sum(),

        (
            patient_safety["patient_risk_level"]
            == "Moderate Review"
        ).sum(),

        (
            patient_safety["patient_risk_level"]
            == "High Review"
        ).sum(),

        (
            patient_safety["patient_risk_level"]
            == "Critical Review"
        ).sum(),

        patient_safety[
            "allergy_review_flag"
        ].sum(),

        (
            ~ranking_safety[
                "disease_compatible"
            ]
        ).sum(),

        ranking_safety[
            "requires_professional_review"
        ].sum(),

        len(no_eligible_recommendations)
    ]
})

display(safety_summary)

,metric,value
0,Total test patients,100
1,Routine-review patients,42
2,Moderate-review patients,52
3,High-review patients,3
4,Critical-review patients,3
5,Patients with recorded-allergy review,22
6,Disease-incompatible recommendations,143
7,Recommendations requiring professional review,69
8,Patients without eligible recommendation,4


In [23]:
assert patient_safety["patient_id"].is_unique

assert not ranking_safety[
    "patient_id"
].isnull().any()

assert not ranking_safety[
    "patient_risk_level"
].isnull().any()

assert (
    ranking_safety.loc[
        ~ranking_safety["disease_compatible"],
        "safety_adjusted_score"
    ] == 0
).all()

assert not final_recommendations[
    "patient_id"
].duplicated().any()

print("All Patient Risk and Safety checks passed.")

All Patient Risk and Safety checks passed.


In [24]:
configuration = {
    "stage": "Patient Risk and Safety Layer",
    "clinical_input": clinical_file,
    "ranking_input": ranking_file,
    "patient_alignment_verified": True,
    "safety_method": (
        "Rule-based patient review flags and "
        "disease-medication-class compatibility"
    ),
    "allergy_limitation": (
        "Recorded allergies cannot be matched to medication "
        "ingredients because ingredient data is unavailable"
    ),
    "output_type": (
        "Experimental medication-class ranking; "
        "not a clinical prescription"
    )
}

with open(configuration_file, "w") as file:
    json.dump(
        configuration,
        file,
        indent=4
    )

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    patient_safety.to_excel(
        writer,
        sheet_name="Patient_Risk_Profiles",
        index=False
    )

    ranking_safety.to_excel(
        writer,
        sheet_name="Safety_Adjusted_Ranking",
        index=False
    )

    final_recommendations.to_excel(
        writer,
        sheet_name="Final_Recommendations",
        index=False
    )

    disease_medicine_map.to_excel(
        writer,
        sheet_name="Disease_Medicine_Map",
        index=False
    )

    no_eligible_recommendations.to_excel(
        writer,
        sheet_name="No_Eligible_Recommendation",
        index=False
    )

    safety_summary.to_excel(
        writer,
        sheet_name="Safety_Summary",
        index=False
    )

    pd.DataFrame(
        list(configuration.items()),
        columns=["configuration", "value"]
    ).to_excel(
        writer,
        sheet_name="Configuration",
        index=False
    )

print("Saved results:", output_file)
print("Saved configuration:", configuration_file)

Saved results: patient_risk_safety_layer_results.xlsx
Saved configuration: patient_risk_safety_configuration.json


In [25]:
for file_name in [
    output_file,
    configuration_file
]:
    print(
        file_name,
        "exists:",
        os.path.exists(file_name)
    )

saved_workbook = pd.ExcelFile(output_file)

print("\nSaved sheets:")
print(saved_workbook.sheet_names)

print("\nFinal status:")
print("Patients evaluated:", len(patient_safety))
print(
    "Final recommendations:",
    len(final_recommendations)
)
print(
    "No eligible recommendation:",
    len(no_eligible_recommendations)
)

patient_risk_safety_layer_results.xlsx exists: True
patient_risk_safety_configuration.json exists: True

Saved sheets:
['Patient_Risk_Profiles', 'Safety_Adjusted_Ranking', 'Final_Recommendations', 'Disease_Medicine_Map', 'No_Eligible_Recommendation', 'Safety_Summary', 'Configuration']

Final status:
Patients evaluated: 100
Final recommendations: 96
No eligible recommendation: 4
